# Experiment 1 — Statistical Analysis

This notebook reports the pre-registered statistical analyses for Experiment 1.  
All models were specified prior to data collection; deviations are noted inline.

**Outcome variables (in order):**
1. Trial Compliance (binary logit)
2. Recognition Accuracy (binary logit)
3. Relatedness Rating (OLS, HC3 robust SEs)
4. Confidence (ordered probit)

**Data scope:** Analyses of trial compliance and accuracy are restricted to trials with a *perceived* memory source (`source == "perceived"`). The *imagined* condition is excluded from these models because it produced near-perfect performance, creating complete separation that prevents logistic regression from converging. The relatedness rating and confidence models use the full dataset with `source` entered as a covariate.

**Centering:** `rating_cen` is mean-centered separately within each analysis dataset (perceived-only vs. full) to preserve interpretability of intercepts.

**Research Questions**
1) How memory and models influence reading hallucinations (failure of trial compliaance)?
2) I want to understand how accuracy changes with respect to the source, memory and model after controlling for reading hallucinations, relatedness rating, and order?
3) How is confidence and related to accuracy, source, memory and model after controlling for reading hallucinations, relatedness rating and order? What is the metacognitive sensitivity (goodman-kruskul coefficient for accuracy and confidence) across memory and model?

---
## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import chi2, norm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.outliers_influence import OLSInfluence
from statsmodels.miscmodels.ordinal_model import OrderedModel
from IPython.display import display, Markdown
import warnings
import rmllm

warnings.filterwarnings('ignore', category=FutureWarning)
np.random.seed(42)

# ── Aesthetics ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', context='paper', font_scale=1.2)
PALETTE = {'perceived': '#4C72B0', 'imagined': '#DD8452',
           'correct': '#55A868',   'incorrect': '#C44E52'}

data_dir = rmllm.config.PROCESSED_DATA_DIR

---
## 1. Data Loading and Descriptives

In [ ]:
df = pd.read_csv(data_dir / 'exp1_trial_data.csv')

print(f"Total trials: {len(df):,}")
print(f"Participants: {df['participant_id'].nunique() if 'participant_id' in df.columns else 'N/A'}")
print()
print("── Source distribution ──")
print(df['source'].value_counts())
print()
print("── Key variable dtypes ──")
print(df[['source', 'memory', 'model', 'order', 'trial_compliance',
          'accuracy', 'rating', 'confidence']].dtypes)
print()
print("── Descriptive statistics ──")
display(df[['trial_compliance', 'accuracy', 'rating', 'confidence']].describe().round(3))



In [ ]:
# ── Reading Hallucination ─────────────────────────────────────────────────────
df["read_hallucination"] = 1 - df["trial_compliance"]

# ── Subset and center ─────────────────────────────────────────────────────────
# Perceived-only subset (used for trial compliance and accuracy models)
df_per = df[df['source'] == 'perceived'].copy()
df_per['rating_cen'] = df_per['rating'] - df_per['rating'].mean()

# Full dataset (used for relatedness rating and confidence models)
df['rating_cen'] = df['rating'] - df['rating'].mean()
df['confidence'] = pd.Categorical(df['confidence'], ordered=True)

print(f"Perceived-only trials: {len(df_per):,}")
print(f"rating_cen grand mean (perceived): {df_per['rating_cen'].mean():.6f}  [should be ~0]")
print(f"rating_cen grand mean (full):      {df['rating_cen'].mean():.6f}  [should be ~0]")

# Note: imagined condition excluded from logit models due to complete separation
df_img = df[df['source'] == 'imagined'].copy()
print(f"\nImagined condition accuracy (excluded from logit models): "
      f"{df_img['accuracy'].mean():.3f} — near-ceiling, causes separation.")

---
## 2. Helper Functions

In [ ]:
def lr_tests(fitted_model, data, formula, model_name='', distr=None):
    """
    Type-II likelihood ratio tests with fallback to Wald tests when
    reduced models fail due to quasi-separation or singularity.
    """
    dep_var, rhs = [s.strip() for s in formula.split('~', 1)]
    # Expand * into main effects + interaction
    expanded = []
    for t in rhs.split('+'):
        t = t.strip()
        if '*' in t:
            a, b = [x.strip() for x in t.split('*')]
            expanded += [a, b, f'{a}:{b}']
        else:
            expanded.append(t)
    terms = []
    seen = set()
    for t in expanded:
        if t not in seen:
            seen.add(t)
            terms.append(t)

    results = []
    for term in terms:
        reduced_terms = [t for t in terms if t != term]
        reduced_formula = f"{dep_var} ~ " + ' + '.join(reduced_terms)
        
        try:
            if distr is not None:  # Ordered probit
                red = OrderedModel.from_formula(
                    reduced_formula, data=data, distr=distr
                ).fit(method='bfgs', disp=False, maxiter=10000)
                lr = 2 * (fitted_model.llf - red.llf)
                ddf = len(fitted_model.params) - len(red.params)
            else:  # Logit
                red = smf.logit(reduced_formula, data=data).fit(
                    disp=False, maxiter=10000, method='lbfgs'
                )
                lr = 2 * (fitted_model.llf - red.llf)
                ddf = len(fitted_model.params) - len(red.params)
            
            p = 1 - chi2.cdf(lr, max(ddf, 1))
            sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else ''
            results.append({
                'Term': term, 
                'LR': round(lr, 3), 
                'df': int(max(ddf, 1)), 
                'p': round(p, 4), 
                '': sig
            })
            
        except Exception as e:
            # Fallback: Wald test from full model (more robust)
            try:
                wald_idx = [i for i, name in enumerate(fitted_model.params.index) 
                           if term in name or (':' in term and term.split(':')[0] in name)]
                if wald_idx:
                    wald_stat = (fitted_model.tvalues.iloc[wald_idx]**2).sum()
                    df_w = len(wald_idx)
                    p_w = 1 - chi2.cdf(wald_stat, df_w)
                    sig = '***' if p_w < .001 else '**' if p_w < .01 else '*' if p_w < .05 else ''
                    results.append({
                        'Term': term, 
                        'LR': np.nan, 
                        'df': df_w, 
                        'p': round(p_w, 4), 
                        '': sig + ' (Wald fallback)'
                    })
                else:
                    results.append({'Term': term, 'LR': np.nan, 'df': np.nan,
                                    'p': np.nan, '': f'ERROR: {str(e)[:60]}'})
            except:
                results.append({'Term': term, 'LR': np.nan, 'df': np.nan,
                                'p': np.nan, '': f'ERROR: {str(e)[:60]}'})

    tbl = pd.DataFrame(results)
    display(Markdown(f'**{model_name} — Likelihood Ratio Tests (Type II)**'))
    display(tbl.style.hide(axis='index'))
    return tbl
def convergence_report(res):
    """Print key convergence diagnostics for a logit result."""
    print(f"  Converged         : {res.mle_retvals.get('converged', 'N/A')}")
    print(f"  Iterations        : {res.mle_retvals.get('iterations', 'N/A')}")
    print(f"  Log-likelihood    : {res.llf:.4f}")
    print(f"  Pseudo-R² (McF.)  : {res.prsquared:.4f}")
    g = res.llf - res.llnull
    p = 1 - chi2.cdf(-2 * res.llnull + 2 * res.llf, res.df_model)
    print(f"  Overall LR χ²({res.df_model:.0f}) : {-2*res.llnull + 2*res.llf:.3f}, p = {p:.4e}")
    # Gradient norm — should be near zero at convergence
    if hasattr(res, 'score_obsv'):
        grad_norm = np.linalg.norm(res.model.score(res.params))
        print(f"  Gradient norm     : {grad_norm:.2e}  [<1e-4 is good]")


def odds_ratios(res, label='Odds Ratios'):
    """Return a DataFrame of ORs with 95 % CIs."""
    coef = res.params
    ci   = res.conf_int()
    tbl  = pd.DataFrame({
        'OR':    np.exp(coef),
        'CI_lo': np.exp(ci[0]),
        'CI_hi': np.exp(ci[1]),
        'p':     res.pvalues
    }).round(3)
    tbl['sig'] = tbl['p'].apply(
        lambda p: '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else ''
    )
    display(Markdown(f'**{label}**'))
    display(tbl)
    return tbl


def avg_marginal_effects(res, label='Average Marginal Effects'):
    """Compute and display AMEs for a logit result."""
    ame = res.get_margeff()
    display(Markdown(f'**{label}**'))
    display(ame.summary_frame().round(4))
    return ame


def partial_eta_sq(anova_tbl):
    """Add partial η² to a statsmodels Type-II ANOVA table."""
    tbl = anova_tbl.copy()
    ss_res = tbl.loc['Residual', 'sum_sq']
    tbl['partial_eta_sq'] = tbl['sum_sq'] / (tbl['sum_sq'] + ss_res)
    tbl.loc['Residual', 'partial_eta_sq'] = np.nan
    return tbl

---
## 3. Model 1 — Trial Compliance

We predicted trial compliance (whether participants followed the task instructions on a given trial) from memory type, model, their interaction, presentation order, and mean-centered relatedness rating. Analysis is restricted to perceived-source trials.

In [ ]:
TC_FORMULA = "trial_compliance ~ C(memory) + C(model) + C(order) + rating_cen"

tc_model = smf.logit(TC_FORMULA, data=df_per).fit(
    disp=False, maxiter=10000, method='lbfgs'
)

print("── Convergence diagnostics ──")
convergence_report(tc_model)

In [ ]:

RH_FORMULA = "read_hallucination ~ C(memory) + C(model) + C(order) + rating_cen"

rh_model = smf.logit(RH_FORMULA, data=df_per).fit(
    disp=False, maxiter=10000, method='lbfgs'
)

print("── Convergence diagnostics ──")
convergence_report(rh_model)

In [ ]:
rh_model.summary2()

### 3.1 Likelihood Ratio Tests (Type II)

In [ ]:
tc_lr = lr_tests(tc_model, df_per, TC_FORMULA, model_name='Trial Compliance')

In [ ]:
rh_lr = lr_tests(rh_model, df_per, RH_FORMULA, model_name='Trial Compliance')

### 3.2 Effect Sizes — Odds Ratios and Average Marginal Effects

In [ ]:
rh_or  = odds_ratios(rh_model, label='Trial Compliance — Odds Ratios (95 % CI)')
rh_ame = avg_marginal_effects(rh_model, label='Trial Compliance — Average Marginal Effects')

### 3.3 Assumption Checks

In [ ]:
# ── Pearson residuals vs fitted ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

fitted_tc  = rh_model.fittedvalues
pearson_tc = rh_model.resid_pearson

axes[0].scatter(fitted_tc, pearson_tc, alpha=0.3, s=15, color='steelblue')
axes[0].axhline(0, color='red', linewidth=1)
axes[0].set_xlabel('Fitted probability')
axes[0].set_ylabel('Pearson residual')
axes[0].set_title('Trial Compliance: Residuals vs Fitted')

# ── Calibration: decile-of-risk plot ──────────────────────────────────────────
cal_df = pd.DataFrame({'obs': df_per['trial_compliance'], 'pred': fitted_tc})
cal_df['decile'] = pd.qcut(cal_df['pred'], q=10, labels=False, duplicates='drop')
cal_grp = cal_df.groupby('decile').agg(mean_pred=('pred','mean'),
                                        mean_obs=('obs','mean')).reset_index()

axes[1].plot([0,1],[0,1], 'k--', linewidth=1, label='Perfect calibration')
axes[1].scatter(cal_grp['mean_pred'], cal_grp['mean_obs'],
                s=60, color='steelblue', zorder=3)
axes[1].set_xlabel('Mean predicted probability')
axes[1].set_ylabel('Observed proportion')
axes[1].set_title('Trial Compliance: Calibration Plot')
axes[1].legend()

plt.tight_layout()
plt.savefig('tc_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Influence: top-5 Cook's distance analogues (deviance residuals) ────────────
deviance_r = rh_model.resid_dev
top5 = np.argsort(np.abs(deviance_r))[-5:]
print("Top-5 observations by |deviance residual|:")
print(df_per.iloc[top5][['memory','model','order','trial_compliance','rating']]
      .assign(deviance_resid=deviance_r[top5]))

---
## 4. Model 2 — Recognition Accuracy

We predicted recognition accuracy from memory type, model, their interaction, trial compliance, presentation order, and mean-centered relatedness rating. Restricted to perceived-source trials.

In [ ]:
ACC_FORMULA = "accuracy ~ C(memory) + C(model) + C(read_hallucination) + C(order) + rating_cen"

model_acc = smf.logit(ACC_FORMULA, data=df_per).fit(disp=False, maxiter=100000, method='lbfgs')
print("── Convergence diagnostics ──")
convergence_report(model_acc)

In [ ]:
model_acc.summary2()

### 4.1 Likelihood Ratio Tests (Type II)

In [ ]:
acc_lr = lr_tests(model_acc, df_per, ACC_FORMULA, model_name='Accuracy')

### 4.2 Effect Sizes — Odds Ratios and Average Marginal Effects

In [ ]:
acc_or  = odds_ratios(model_acc, label='Accuracy — Odds Ratios (95 % CI)')
acc_ame = avg_marginal_effects(model_acc, label='Accuracy — Average Marginal Effects')

### 4.3 Assumption Checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

fitted_acc  = model_acc.fittedvalues
pearson_acc = model_acc.resid_pearson

axes[0].scatter(fitted_acc, pearson_acc, alpha=0.3, s=15, color='#55A868')
axes[0].axhline(0, color='red', linewidth=1)
axes[0].set_xlabel('Fitted probability')
axes[0].set_ylabel('Pearson residual')
axes[0].set_title('Accuracy: Residuals vs Fitted')

cal_df2 = pd.DataFrame({'obs': df_per['accuracy'], 'pred': fitted_acc})
cal_df2['decile'] = pd.qcut(cal_df2['pred'], q=10, labels=False, duplicates='drop')
cal_grp2 = cal_df2.groupby('decile').agg(mean_pred=('pred','mean'),
                                          mean_obs=('obs','mean')).reset_index()

axes[1].plot([0,1],[0,1], 'k--', linewidth=1, label='Perfect calibration')
axes[1].scatter(cal_grp2['mean_pred'], cal_grp2['mean_obs'],
                s=60, color='#55A868', zorder=3)
axes[1].set_xlabel('Mean predicted probability')
axes[1].set_ylabel('Observed proportion')
axes[1].set_title('Accuracy: Calibration Plot')
axes[1].legend()

plt.tight_layout()
plt.savefig('acc_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

deviance_a = model_acc.resid_dev
top5a = np.argsort(np.abs(deviance_a))[-5:]
print("Top-5 observations by |deviance residual|:")
print(df_per.iloc[top5a][['memory','model','order','accuracy','rating']]
      .assign(deviance_resid=deviance_a[top5a]))

### 4.4 Gemma3:12b — Accuracy Conditional on Reading Hallucination, by Memory

Descriptive check isolating source-judgment ability from encoding (word-reproduction) fidelity for Gemma3:12b, whose accuracy declined under Trial-Chain despite a sharp drop in its own reading-hallucination rate.

In [ ]:
# Gemma3:12b: accuracy conditional on reading-hallucination status, by memory
sub = df_per[df_per['model'] == 'Gemma3:12b']
cond_acc = (sub.groupby(['memory', 'read_hallucination'])['accuracy']
               .agg(n='size', accuracy='mean')
               .round(3))
print(cond_acc)

### Manuscript cross-reference — Gemma3:12b conditional accuracy

Results from the cell above are reported in `main.tex` under **§ Accuracy: Ceiling-Level Performance for Internal Items, Variable Performance for External Items** (added 2026-07-10).

Reported in text as: "conditional on correct word reproduction (no reading hallucination), source-judgment accuracy fell from 100% (n = 53) in Single-Turn to 53.5% (n = 129) in Trial-Chain."

In [ ]:
RR_FORMULA = ("rating_cen ~ C(accuracy) + C(source) + C(memory) + C(model) "
              "+ C(read_hallucination) + C(order)")

model_rr = smf.ols(RR_FORMULA, data=df).fit(cov_type='HC3')

print(f"R²        : {model_rr.rsquared:.4f}")
print(f"Adj. R²   : {model_rr.rsquared_adj:.4f}")
print(f"F({model_rr.df_model:.0f}, {model_rr.df_resid:.0f}) = {model_rr.fvalue:.3f}, p = {model_rr.f_pvalue:.4e}")
model_rr.summary2()

### 5.1 Type II ANOVA with Partial η²

> **Note:** `anova_lm` uses OLS fit without robust SEs for the F-tests. The coefficient-level results above (with HC3 SEs) are primary; the ANOVA table is provided for effect-size reporting.

In [ ]:
# Refit without robust SEs for anova_lm (which requires standard OLS)
model_rr_ols = smf.ols(RR_FORMULA, data=df).fit()
anova_rr = anova_lm(model_rr_ols, typ=2)
anova_rr_eta = partial_eta_sq(anova_rr)
display(Markdown('**Relatedness Rating — Type II ANOVA + Partial η²**'))
display(anova_rr_eta.round(4))

### 5.2 Assumption Checks

In [ ]:
infl = OLSInfluence(model_rr_ols)
cooks_d, _ = infl.cooks_distance

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residuals vs fitted
axes[0].scatter(model_rr_ols.fittedvalues, model_rr_ols.resid,
                alpha=0.3, s=15, color='#4C72B0')
axes[0].axhline(0, color='red', linewidth=1)
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# Q-Q plot
from scipy.stats import probplot
probplot(model_rr_ols.resid, plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

# Cook's distance
axes[2].stem(np.arange(len(cooks_d)), cooks_d, markerfmt='C1o',
             linefmt='C1-', basefmt='k-')
threshold = 4 / len(df)
axes[2].axhline(threshold, color='red', linestyle='--',
                label=f"4/n = {threshold:.4f}")
axes[2].set_xlabel('Observation index')
axes[2].set_ylabel("Cook's distance")
axes[2].set_title("Influence: Cook's Distance")
axes[2].legend()

plt.tight_layout()
plt.savefig('rr_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

n_influential = (cooks_d > threshold).sum()
print(f"Observations exceeding Cook's distance threshold (4/n={threshold:.4f}): {n_influential}")

---
## 6. Model 4 — Confidence (Ordered Probit)

Confidence ratings were modeled using an ordered probit regression with source, accuracy, model, trial compliance, mean-centered relatedness rating, and presentation order as predictors.

In [ ]:
CONF_FORMULA = ("confidence ~ C(source) + C(accuracy) + C(memory) + C(model) + C(read_hallucination) + rating_cen + C(order)")

mod_prob = OrderedModel.from_formula(CONF_FORMULA, df, distr='probit')
res_prob = mod_prob.fit(method='bfgs', maxiter=10000)

print(f"Converged: {res_prob.mle_retvals.get('converged', 'N/A')}")
print(f"Log-likelihood: {res_prob.llf:.4f}")
res_prob.summary()

### Manuscript cross-reference — Exp 1 Confidence Ordered Probit

Results from this cell are reported in `main.tex` under
**§ Confidence in Single-Trial Reality Monitoring** (added 2026-06-08).

Key coefficients (ordered probit, combined sources):

| Predictor | b | z | p |
|---|---|---|---|
| source (perceived) | 1.803 | 22.677 | < .001 |
| reading hallucination | −0.666 | −5.469 | < .001 |
| accuracy (correct) | 1.047 | 8.470 | < .001 |
| memory (Trial-Chain) | 1.279 | 19.400 | < .001 |

These are from `res_prob` (Cell 34 above). Source perceived b=+1.80 means
perceived-source trials elicit higher confidence than imagined-source trials.
Unlike Experiment 2, trial-level accuracy IS a significant confidence predictor here.

### 6.1 Likelihood Ratio Tests (Type II)

In [ ]:
# Formula passed to lr_tests must match the fitted model exactly
conf_lr = lr_tests(
    res_prob, df, CONF_FORMULA,
    model_name='Confidence (Ordered Probit)',
    distr='probit'
)

### 6.2 Effect Sizes — Marginal Probabilities for Key Predictors

For ordered probit models we report the change in predicted probability for each confidence category associated with moving from the reference to the comparison level of each predictor.

In [ ]:
def ordinal_marginal_probs(res, data, var, levels, formula, distr='probit'):
    """
    Compute mean predicted probability for each confidence category
    at each level of `var`, holding other predictors at observed values.
    """
    cats = sorted(data['confidence'].cat.categories)
    rows = []
    for lvl in levels:
        d_tmp = data.copy()
        d_tmp[var] = lvl
        probs = res.predict(d_tmp)
        if hasattr(probs, 'values'):
            probs = probs.values
        mean_probs = probs.mean(axis=0)
        for cat, mp in zip(cats, mean_probs):
            rows.append({var: lvl, 'Confidence': cat, 'Mean P': round(mp, 4)})
    tbl = pd.DataFrame(rows).pivot(index=var, columns='Confidence', values='Mean P')
    return tbl

# Marginal probabilities for source and accuracy
for var, levels in [('source', df['source'].unique()),
                    ('accuracy', df['accuracy'].unique())]:
    try:
        tbl = ordinal_marginal_probs(res_prob, df, var, levels, CONF_FORMULA)
        display(Markdown(f'**Confidence marginal probabilities by `{var}`**'))
        display(tbl.round(4))
    except Exception as e:
        print(f"Could not compute for {var}: {e}")

# 7. Metacognition

In [ ]:
from rmllm import gamma

In [ ]:
df.columns

In [ ]:
gamma_results = gamma.calculate_gamma_across_groups(df, ["model", "memory"])
gamma_results

In [ ]:
gamma_src_results = gamma.calculate_gamma_across_groups(df, ["model", "memory","source"])
gamma_src_results

---
# 8. Metacognitive Sensitivity γ — Fisher-Z Analysis

Goodman–Kruskal γ in [-1, 1] is Fisher-Z transformed: **γ_z = arctanh(γ)** (clipped at ±0.9999).

> **Sample size note.** In Exp 1, each trace × model × memory cell has exactly one trial, so γ must be computed at the **(model × memory) group level** (288 trials/cell, 144 per source). This yields 12 groups of which **n = 7 are valid** — 5 are NaN because some model/memory conditions show constant accuracy (floor/ceiling), making γ undefined. All formal inference below is therefore **exploratory**.

In [ ]:
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from rmllm import gamma as gamma_mod

def fisher_z(g):
    """arctanh transform, clipped to ±0.9999 to avoid ±inf."""
    return np.arctanh(np.clip(np.asarray(g, dtype=float), -0.9999, 0.9999))

# ── Compute γ at (model × memory) and (model × memory × source) ──────────────
g_mm  = gamma_mod.calculate_gamma_across_groups(df, ["model", "memory"])
g_mms = gamma_mod.calculate_gamma_across_groups(df, ["model", "memory", "source"])

g_mm["gamma_z"]  = fisher_z(g_mm["gamma"])
g_mms["gamma_z"] = fisher_z(g_mms["gamma"])

print(f"Valid γ: {g_mm['gamma'].notna().sum()} / {len(g_mm)}")
g_mm[["model","memory","gamma","gamma_z","n_trials"]]

## 8.1 Source breakdown (model × memory × source)

In [ ]:
g_mms[['model','memory','source','gamma','gamma_z']]

## 8.2 Summary pivot table

In [ ]:
summary = g_mms.pivot_table(
    index=["model", "memory"], columns="source",
    values=["gamma", "gamma_z"]
).round(4)
display(summary)

## 8.3 Formal tests (n = 7, exploratory)

Given the small sample, only the **memory** factor can be tested (SingleTurn vs TrialChain). A full model including *model* is over-parameterised at n = 7.

In [ ]:
valid = g_mm.dropna(subset=["gamma_z"]).copy()
st = valid.loc[valid["memory"]=="SingleTurn", "gamma_z"]
tc = valid.loc[valid["memory"]=="TrialChain",  "gamma_z"]

print(f"SingleTurn  n={len(st)}  mean={st.mean():.4f}  SD={st.std():.4f}")
print(f"TrialChain  n={len(tc)}  mean={tc.mean():.4f}  SD={tc.std():.4f}")

# One-sample t: H₀ mean γ_z = 0
t0, p0 = stats.ttest_1samp(valid["gamma_z"], 0)
print(f"\nH₀: mean γ_z = 0  →  t({len(valid)-1})={t0:.3f}  p={p0:.4f}")

# OLS: γ_z ~ memory
m1 = smf.ols("gamma_z ~ C(memory, Treatment('SingleTurn'))", data=valid).fit()
lr1   = 2*(m1.llf - smf.ols("gamma_z ~ 1", data=valid).fit().llf)
p_lr1 = stats.chi2.sf(lr1, df=1)
print(f"LR test memory: χ²(1)={lr1:.4f}  p={p_lr1:.4f}")
print()
m1.summary2()

In [ ]:
# Paired subset: models with data in BOTH memory conditions
complete = (valid.groupby("model")["memory"].nunique()
            .loc[lambda s: s==2].index.tolist())
paired = valid[valid["model"].isin(complete)].copy()
print(f"Paired models: {complete}  (n={len(paired)})")

if len(paired) >= 4:
    m2 = smf.ols("gamma_z ~ C(memory, Treatment('SingleTurn')) + C(model)",
                 data=paired).fit()
    lr2   = 2*(m2.llf - smf.ols("gamma_z ~ 1", data=paired).fit().llf)
    p_lr2 = stats.chi2.sf(lr2, df=int(m2.df_model))
    print(f"LR test (memory+model vs null): χ²({int(m2.df_model)})={lr2:.4f}  p={p_lr2:.4f}")
    display(m2.summary2())

## 8.4 Figure: raw γ and γ_z by model × memory

In [ ]:
import matplotlib.pyplot as plt

MODEL_ORDER = ["Gemma3:12b","Gemma3:12b-QAT","Gemma3:27b","Gemma3:27b-QAT",
               "Llama4:16x17b","Llama3.3:70b"]
PAL = {"SingleTurn":"#2196F3","TrialChain":"#FF9800"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
fig.suptitle("Exp 1 — Metacognitive γ: raw and Fisher-Z transformed", fontsize=12)

for ax, col, ylabel in zip(axes, ["gamma","gamma_z"], ["γ (raw)","γ_z = arctanh(γ)"]):
    for mem, grp in g_mm.groupby("memory"):
        v = grp.set_index("model").reindex(MODEL_ORDER)[col].values
        ax.plot(range(len(MODEL_ORDER)), v, "o-", label=mem,
                color=PAL[mem], linewidth=1.8, markersize=7)
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_xticks(range(len(MODEL_ORDER)))
    ax.set_xticklabels(MODEL_ORDER, rotation=38, ha="right", fontsize=8)
    ax.set_ylabel(ylabel); ax.set_title(ylabel)
    ax.legend(title="Memory", fontsize=8)

plt.tight_layout()
plt.savefig(rmllm.config.REPORTS_DIR / "figures" / "exp1_gamma_z.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

---
## 7. Key Results Figure

2 rows and columns panel:

1.1 Reading Hallucination, axis => x = model, y = reading hallucination; hue = conversation memory.

1.2 Accuracy, axis => x = model, y = accuracy (perceived/external); hue = conversation memory.

1.3 Confidence,  axis => x = confidence, y = proportion of trials; linecolor/hue = model, line style = based on source. Marginal bold black solid and dotted line for accuracy, another bold marginal line with different color for source as solid and dotted line.

2.1 Fisher's Z transformed metacognitive sensitivity. axis x = model, y = Fisher's Z transformed gamma coeffiecient; hue = memory (single turn and trial chain) 

2.2 Fisher's Z transformed metacognitive sensitivity. axis x = model, y = Fisher's Z transformed gamma coeffiecient; hue = memory (single turn and trial chain).

2.3 Relatedness rating. axis x = model, y = relatedness rating; hue = memory (single turn and trial chain), line style = source (imagined and perceived).

In [ ]:
# ============================================================
# Section 7 — Key Results Figure
# Paste each fenced block into its own notebook cell.
# All variables (df, df_per, gamma_results, gamma_src_results,
# tc_model, rh_model, model_acc, model_rr, res_prob, PALETTE)
# are assumed to be defined by earlier cells in the notebook.
# ============================================================


# ── CELL 1 ── imports & shared aesthetics ───────────────────────────────────
# (run once; safe to re-run)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.stats import norm as _snorm
from scipy.special import expit

# ── Ordered model / colour constants ──────────────────────────────────────
MODEL_ORDER = [
    'Gemma3:12b', 'Gemma3:12b-QAT', 'Gemma3:27b',
    'Gemma3:27b-QAT', 'Llama3.3:70b', 'Llama4:16x17b',
]
MODEL_LABELS = [
    'G3:12b', 'G3:12b\nQAT', 'G3:27b',
    'G3:27b\nQAT', 'L3.3:70b', 'L4:16x17b',
]

MEM_PALETTE = {'SingleTurn': '#4C72B0', 'TrialChain': '#DD8452'}
SRC_LS      = {'perceived': '-', 'imagined': '--'}
MEM_LW      = {'SingleTurn': 2.0, 'TrialChain': 2.0}
CONF_COLORS = plt.cm.tab10.colors   # one per model

plt.rcParams.update({
    # ── Font ──────────────────────────────────────────────
    'font.family'        : 'sans-serif',
    'font.size'          : 11,
    'axes.titlesize'     : 12,
    'axes.titleweight'   : 'bold',
    'axes.labelsize'     : 11,
    'axes.labelweight'   : 'bold',
    'xtick.labelsize'    : 10,
    'ytick.labelsize'    : 10,
    # ── Axes / spines ──────────────────────────────────────
    'axes.linewidth'     : 1.0,
    'axes.facecolor'     : 'white',
    'figure.facecolor'   : 'white',
    'axes.grid'          : False,
    # ── Ticks ──────────────────────────────────────────────
    'xtick.bottom'       : True,
    'xtick.top'          : False,
    'ytick.left'         : True,
    'ytick.right'        : False,
    'xtick.major.size'   : 5,
    'ytick.major.size'   : 5,
    'xtick.minor.size'   : 2.5,
    'ytick.minor.size'   : 2.5,
    'xtick.major.width'  : 1.0,
    'ytick.major.width'  : 1.0,
    'xtick.direction'    : 'out',
    'ytick.direction'    : 'out',
    # ── Legend ─────────────────────────────────────────────
    'legend.fontsize'    : 9,
    'legend.title_fontsize': 9,
    'legend.framealpha'  : 0.9,
    'legend.edgecolor'   : '#cccccc',
    # ── Lines / markers ────────────────────────────────────
    'lines.linewidth'    : 1.5,
    'lines.markersize'   : 6,
})

print("Aesthetics loaded.")


# ── CELL 2 ── helper functions ──────────────────────────────────────────────

def fisher_z(g):
    """Fisher's Z-transform for Goodman-Kruskal gamma; NaN-safe."""
    g = np.clip(g, -0.9999, 0.9999)
    return 0.5 * np.log((1.0 + g) / (1.0 - g))


def _obs_proportions(df_in, groupby_cols, outcome_col):
    """
    Return a dict: {group_key -> Series} of observed proportions
    of outcome_col values within each group.
    """
    result = {}
    for key, grp in df_in.groupby(groupby_cols):
        counts = grp[outcome_col].value_counts(normalize=True).sort_index()
        result[key] = counts
    return result


def _bar_panel(ax, data_dict, models, model_labels,
               memories, mem_palette, bar_w=0.38,
               ylabel='', fmt_pct=False, ylim=(0, 1)):
    """Generic grouped bar plot for panels A and B."""
    x = np.arange(len(models))
    offsets = [-bar_w / 2, bar_w / 2]
    for i, mem in enumerate(memories):
        vals = [data_dict.get((m, mem), np.nan) for m in models]
        ax.bar(x + offsets[i], vals, width=bar_w,
               color=mem_palette[mem], label=mem,
               alpha=0.88, edgecolor='white', linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, fontsize=10, fontweight='bold',
                       rotation=45, ha='right', rotation_mode='anchor')
    ax.set_xlim(-0.6, len(models) - 0.4)
    ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
    ax.set_ylim(*ylim)
    ax.tick_params(axis='x', which='major', bottom=True,
                   length=5, width=1.0, direction='out')
    ax.tick_params(axis='y', which='major', left=True,
                   length=5, width=1.0, direction='out', labelsize=10)
    if fmt_pct:
        ax.yaxis.set_major_formatter(
            plt.FuncFormatter(lambda v, _: f'{v:.0%}'))


def _gamma_panel(ax, gamma_dict, models, model_labels,
                 memories, mem_palette, bar_w=0.38,
                 ylabel="Fisher's Z (γ)"):
    """Generic grouped bar plot for gamma panels (D and E)."""
    x = np.arange(len(models))
    offsets = [-bar_w / 2, bar_w / 2]
    for i, mem in enumerate(memories):
        for xi, m in enumerate(models):
            g_raw = gamma_dict.get((m, mem), np.nan)
            fz    = fisher_z(g_raw) if not np.isnan(g_raw) else np.nan
            color = mem_palette[mem]
            if not np.isnan(fz):
                ax.bar(xi + offsets[i], fz, width=bar_w,
                       color=color, alpha=0.88,
                       edgecolor='white', linewidth=0.6,
                       label=mem if xi == 0 else '')
    ax.axhline(0, color='black', linewidth=1.0, linestyle='--', alpha=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, fontsize=10, fontweight='bold',
                       rotation=45, ha='right', rotation_mode='anchor')
    ax.set_xlim(-0.6, len(models) - 0.4)
    ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', which='major', bottom=True,
                   length=5, width=1.0, direction='out')
    ax.tick_params(axis='y', which='major', left=True,
                   length=5, width=1.0, direction='out', labelsize=10)


def _annotate_panel(ax, letter, title):
    """Bold panel letter (top-left) + bold title."""
    ax.text(-0.15, 1.08, letter, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top', ha='left')
    ax.set_title(title, pad=8, fontsize=12, fontweight='bold')


print("Helper functions defined.")


# ── CELL 3 ── prepare per-panel data from live notebook variables ────────────
# Requires: df, df_per, gamma_results, gamma_src_results

# ── A/B: observed means ────────────────────────────────────────────────────
rh_means  = {}
acc_means = {}
for (m, mem), grp in df_per.groupby(['model', 'memory']):
    rh_means[(m, mem)]  = grp['read_hallucination'].mean()
    acc_means[(m, mem)] = grp['accuracy'].mean()

# ── C: confidence distribution ─────────────────────────────────────────────
conf_obs = {}
conf_levels_all = sorted(
    df['confidence'].dropna().astype(float).astype(int).unique()
)
for (m, src, mem), grp in df.groupby(['model', 'source', 'memory']):
    counts = (grp['confidence'].astype(float).astype(int)
                .value_counts(normalize=True))
    conf_obs[(m, src, mem)] = counts.reindex(conf_levels_all, fill_value=0.0)

# marginal by source
conf_src_marginal = {}
for src in ('perceived', 'imagined'):
    rows = [v for (m, s, mem), v in conf_obs.items() if s == src]
    conf_src_marginal[src] = pd.concat(rows, axis=1).mean(axis=1)

# marginal by accuracy
conf_acc_marginal = {}
for acc_val in (0, 1):
    rows = []
    for (m, src, mem), grp in df.groupby(['model', 'source', 'memory']):
        sub = grp[grp['accuracy'] == acc_val]
        if len(sub) > 0:
            counts = (sub['confidence'].astype(float).astype(int)
                        .value_counts(normalize=True))
            rows.append(counts.reindex(conf_levels_all, fill_value=0.0))
    if rows:
        conf_acc_marginal[acc_val] = pd.concat(rows, axis=1).mean(axis=1)

# ── D: gamma — all sources ─────────────────────────────────────────────────
gamma_all_dict = {
    (row['model'], row['memory']): row['gamma']
    for _, row in gamma_results.iterrows()
}

# ── E: gamma — perceived only ──────────────────────────────────────────────
gamma_per_dict = {
    (row['model'], row['memory']): row['gamma']
    for _, row in gamma_src_results[
        gamma_src_results['source'] == 'perceived'
    ].iterrows()
}

# ── F: relatedness rating ──────────────────────────────────────────────────
rr_means = {}
rr_se    = {}
for (m, src, mem), grp in df.groupby(['model', 'source', 'memory']):
    rr_means[(m, src, mem)] = grp['rating'].mean()
    rr_se[(m, src, mem)]    = grp['rating'].std() / np.sqrt(len(grp))

print("Panel data prepared.")


# ── CELL 4 ── draw the figure ────────────────────────────────────────────────

MEMORIES = ['SingleTurn', 'TrialChain']

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(
    2, 3, figure=fig,
    hspace=0.65, wspace=0.42,
    left=0.07, right=0.97, top=0.92, bottom=0.14
)

x = np.arange(len(MODEL_ORDER))

# ── Panel A: Reading Hallucination ─────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
_bar_panel(ax_a, rh_means, MODEL_ORDER, MODEL_LABELS, MEMORIES, MEM_PALETTE,
           ylabel='Reading Hallucination Rate', fmt_pct=False, ylim=(0, 1))
ax_a.legend(title='Memory', loc='upper left', framealpha=0.9)
_annotate_panel(ax_a, 'A', 'Reading Hallucination')

# ── Panel B: Accuracy ──────────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
_bar_panel(ax_b, acc_means, MODEL_ORDER, MODEL_LABELS, MEMORIES, MEM_PALETTE,
           ylabel='Accuracy', fmt_pct=False, ylim=(0, 1))
ax_b.legend(title='Memory', loc='lower right', framealpha=0.9)
_annotate_panel(ax_b, 'B', 'Recognition Accuracy (Perceived Source)')

# ── Panel C: Confidence Distribution ──────────────────────────────────────
ax_c = fig.add_subplot(gs[0, 2])

# per-model lines (perceived × SingleTurn)
for mi, model in enumerate(MODEL_ORDER):
    key = (model, 'perceived', 'SingleTurn')
    if key in conf_obs:
        probs = conf_obs[key]
        ax_c.plot(probs.index, probs.values,
                  color=CONF_COLORS[mi], linewidth=1.4, alpha=0.75,
                  label=MODEL_LABELS[mi].replace('\n', ' '))

# accuracy marginals — black, * markers
for acc_val, ls, label in [(1, '-', 'Acc · correct'), (0, ':', 'Acc · incorrect')]:
    if acc_val in conf_acc_marginal:
        probs = conf_acc_marginal[acc_val]
        ax_c.plot(probs.index, probs.values,
                  color='black', ls=ls, linewidth=1.4,
                  marker='*', markersize=8, zorder=11, label=label)

# source marginals — dark grey, + markers
for src, ls, label in [('perceived', '-', 'Src · perceived'),
                        ('imagined',  ':', 'Src · imagined')]:
    probs = conf_src_marginal[src]
    ax_c.plot(probs.index, probs.values,
              color='#555555', ls=ls, linewidth=1.4,
              marker='P', markersize=6, zorder=10, label=label)

ax_c.set_xlabel('Confidence Level', fontsize=11, fontweight='bold')
ax_c.set_ylabel('Proportion of Trials', fontsize=11, fontweight='bold')
ax_c.set_xticks(conf_levels_all)
ax_c.tick_params(axis='x', which='major', bottom=True,
                 length=5, width=1.0, direction='out', labelsize=10)
ax_c.tick_params(axis='y', which='major', left=True,
                 length=5, width=1.0, direction='out', labelsize=10)
ax_c.legend(ncol=2, fontsize=8, loc='upper left', framealpha=0.9)
_annotate_panel(ax_c, 'C', 'Confidence Distribution')

# ── Panel D: Gamma — All Sources ──────────────────────────────────────────
ax_d = fig.add_subplot(gs[1, 0])
_gamma_panel(ax_d, gamma_all_dict, MODEL_ORDER, MODEL_LABELS, MEMORIES, MEM_PALETTE)
ax_d.set_ylim(-1.8, 3.8)
handles_d = [
    plt.Rectangle((0,0),1,1, color=MEM_PALETTE['SingleTurn'],
                  alpha=0.88, label='SingleTurn'),
    plt.Rectangle((0,0),1,1, color=MEM_PALETTE['TrialChain'],
                  alpha=0.88, label='TrialChain'),
]
ax_d.legend(handles=handles_d, title='Memory', loc='upper left', framealpha=0.9)
_annotate_panel(ax_d, 'D', 'Metacognitive Sensitivity (All Sources)')

# ── Panel E: Gamma — Perceived Only ───────────────────────────────────────
ax_e = fig.add_subplot(gs[1, 1])
_gamma_panel(ax_e, gamma_per_dict, MODEL_ORDER, MODEL_LABELS, MEMORIES, MEM_PALETTE)
ax_e.set_ylim(-1.8, 4.8)
handles_e = [
    plt.Rectangle((0,0),1,1, color=MEM_PALETTE['SingleTurn'],
                  alpha=0.88, label='SingleTurn'),
    plt.Rectangle((0,0),1,1, color=MEM_PALETTE['TrialChain'],
                  alpha=0.88, label='TrialChain'),
]
ax_e.legend(handles=handles_e, title='Memory', loc='upper left', framealpha=0.9)
_annotate_panel(ax_e, 'E', 'Metacognitive Sensitivity (Perceived Source)')

# ── Panel F: Relatedness Rating ────────────────────────────────────────────
ax_f = fig.add_subplot(gs[1, 2])

for mem in MEMORIES:
    for src in ('perceived', 'imagined'):
        means = [rr_means.get((m, src, mem), np.nan) for m in MODEL_ORDER]
        ses   = [rr_se.get(  (m, src, mem), 0.0)    for m in MODEL_ORDER]
        ax_f.errorbar(
            x, means, yerr=ses,
            color=MEM_PALETTE[mem],
            ls=SRC_LS[src],
            lw=MEM_LW[mem],
            marker='o', markersize=5,
            capsize=3, alpha=0.88,
        )

ax_f.set_xticks(x)
ax_f.set_xticklabels(MODEL_LABELS, fontsize=10, fontweight='bold',
                     rotation=45, ha='right', rotation_mode='anchor')
ax_f.set_xlim(-0.6, len(MODEL_ORDER) - 0.4)
ax_f.set_ylabel('Relatedness Rating (Mean ± SE)', fontsize=11, fontweight='bold')
ax_f.set_ylim(20, 100)
ax_f.tick_params(axis='x', which='major', bottom=True,
                 length=5, width=1.0, direction='out')
ax_f.tick_params(axis='y', which='major', left=True,
                 length=5, width=1.0, direction='out', labelsize=10)

legend_lines_f = [
    Line2D([0],[0], color=MEM_PALETTE['SingleTurn'], lw=2.0,
           ls='-',  label='SingleTurn · perceived'),
    Line2D([0],[0], color=MEM_PALETTE['SingleTurn'], lw=2.0,
           ls='--', label='SingleTurn · imagined'),
    Line2D([0],[0], color=MEM_PALETTE['TrialChain'], lw=2.0,
           ls='-',  label='TrialChain · perceived'),
    Line2D([0],[0], color=MEM_PALETTE['TrialChain'], lw=2.0,
           ls='--', label='TrialChain · imagined'),
]
ax_f.legend(handles=legend_lines_f, title='Memory · Source',
            fontsize=8, loc='lower right', framealpha=0.9)
_annotate_panel(ax_f, 'F', 'Relatedness Rating')

# ── Title & save ───────────────────────────────────────────────────────────
fig.suptitle('Experiment 1 — Key Results',
             fontsize=14, fontweight='bold', y=0.98)

plt.savefig('exp1_key_results.pdf', dpi=300, bbox_inches='tight')
plt.savefig('exp1_key_results.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved as exp1_key_results.pdf / .png")

---
## 8. Summary of Results

| Model | Method | Key effects | Effect size |
|---|---|---|---|
| Trial Compliance | Logit (MLE) | See LR table above | ORs in §3.2 |
| Accuracy | Logit (MLE) | See LR table above | ORs in §4.2 |
| Relatedness Rating | OLS (HC3) | See ANOVA table above | Partial η² in §5.1 |
| Confidence | Ordered Probit | See LR table above | Marginal probs in §6.2 |

**Exclusions:** Imagined-condition trials (`source == "imagined"`) were excluded from Models 1 and 2 because near-perfect performance in this condition creates complete separation, preventing logistic regression from converging to finite parameter estimates. All imagined trials are retained in Models 3 and 4, with `source` entered as a predictor.

**Software:** Analyses were conducted in Python using `statsmodels` (v0.x) and `scipy` (v1.x). All code is available in this notebook.